In [1]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
from scipy.spatial import cKDTree

print(f"CUDA Available: {torch.cuda.is_available()}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
CUDA Available: True


In [2]:
# --- DATA PATHS ---
base_path = "data/global_view/"
room_prefix = "Area_1_conferenceRoom_1"
anno_path = "data/Stanford3dDataset/Stanford3dDataset_v1.2_Aligned_Version/Area_1/conferenceRoom_1/Annotations/"
total_samples = 50

In [3]:
# --- LOAD AND MERGE .PLY SAMPLES ---
full_conference_room = o3d.geometry.PointCloud()

for i in range(total_samples):
    file_name = f"{room_prefix}_sample{i}_simCtr_{i}.ply"
    file_path = os.path.join(base_path, file_name)

    if os.path.exists(file_path):
        current_sample = o3d.io.read_point_cloud(file_path)
        full_conference_room += current_sample

        if i % 10 == 0:
            print(f"Progress: Combined Sample {i} / {total_samples - 1}" )
    
    else:
        print(f"{file_name} not found.")

full_room_cleaned = full_conference_room.voxel_down_sample(voxel_size = 0.02)

Progress: Combined Sample 0 / 49
Area_1_conferenceRoom_1_sample8_simCtr_8.ply not found.
Progress: Combined Sample 10 / 49
Progress: Combined Sample 20 / 49
Area_1_conferenceRoom_1_sample23_simCtr_23.ply not found.
Area_1_conferenceRoom_1_sample30_simCtr_30.ply not found.
Area_1_conferenceRoom_1_sample31_simCtr_31.ply not found.
Area_1_conferenceRoom_1_sample32_simCtr_32.ply not found.
Area_1_conferenceRoom_1_sample35_simCtr_35.ply not found.
Progress: Combined Sample 40 / 49
Area_1_conferenceRoom_1_sample45_simCtr_45.ply not found.


In [4]:
# --- DEFINE LABELS AND PALETTE ---
s3dis_labels = {
    'ceiling': 0, 'floor': 1, 'wall': 2, 'beam': 3, 'column': 4,
    'window': 5, 'door': 6, 'table': 7, 'chair': 8, 'sofa': 9, 
    'bookcase': 10, 'board': 11, 'clutter': 12
}

palette = np.array([
    [0, 1, 0],      # Green -> Ceiling
    [0, 0, 1],      # Blue -> Floor
    [0, 1, 1],      # Cyan -> Wall
    [1, 1, 0],      # Yellow -> Beam
    [1, 0, 1],      # Magenta -> Column
    [0.5, 0.5, 0.5],# Grey -> Window
    [0.5, 0, 0],    # Maroon -> Door
    [1, 0.5, 0],    # Orange -> Table
    [1, 0, 0],      # Red -> Chair
    [0.6, 0.3, 0.1],# Brown -> Sofa
    [0.5, 0.5, 0],  # Olive -> Bookcase
    [0, 0.5, 0.5],  # Teal -> Board
    [0.2, 0.2, 0.2] # Dark Grey -> Clutter
])

o3d.visualization.draw_geometries([full_room_cleaned])

In [5]:
# --- ANNOTATION FUNCTION ---
def load_annotations(path):
    all_points = []
    all_labels = []

    for file in os.listdir(path):
        if file.endswith('.txt'):
            class_name = file.split('_')[0]
            label_idx = s3dis_labels.get(class_name, 12)

            # Load only the first 3 columns (x, y, z)
            data = np.loadtxt(os.path.join(path, file))
            points = data[:, :3]

            all_points.append(points)
            all_labels.append(np.full(len(points), label_idx))
    
    # Return the combined data so we can use it outside
    return np.vstack(all_points), np.concatenate(all_labels)

# Load the ground truth data
gt_points, gt_labels = load_annotations(anno_path)

In [6]:
# --- MAPPING THE .PLY TO THE .TXT LABELS ---
tree = cKDTree(gt_points)

# Find the nearest neighbour in the .txt data for each point in the .ply data
_, indices = tree.query(np.asarray(full_room_cleaned.points), k=1)
final_labels = gt_labels[indices]

# Apply the palette to the colors
full_room_cleaned.colors = o3d.utility.Vector3dVector(palette[final_labels])

In [7]:
# --- VISUALIZE ---
o3d.visualization.draw_geometries([full_room_cleaned])